# Exercise 34 — Scope: local vs global

## Concept

When Python sees a name, it looks it up in scopes in this order (LEGB):

1. **L**ocal — inside the current function
2. **E**nclosing — outer function(s) if nested
3. **G**lobal — module level
4. **B**uilt-in — `print`, `len`, etc.

```python
x = 10                       # global

def show():
    print(x)                 # reads global x → 10

def shadow():
    x = 99                   # NEW local — does not touch global
    print(x)                 # 99

show()                       # 10
shadow()                     # 99
print(x)                     # 10 — unchanged
```

### Assignment makes it local

The moment Python sees `x = ...` anywhere in a function body, `x` is treated as local **throughout that function**. So this fails:

```python
count = 0

def bump():
    count = count + 1        # UnboundLocalError — `count` is local here, but unbound at the read
```

To really write to a global, use `global` (works, but generally a smell):

```python
def bump():
    global count
    count = count + 1
```

### Why mutating globals from inside functions is bad

It couples otherwise-independent functions, makes testing harder, and produces "action at a distance" bugs (changing module-level state in one function silently breaks another). Prefer passing values **in** as arguments and returning new values **out**.

## Task 1 — Predict, then verify

Define the functions below and call them. Before running, write down what you think each `print` will produce. Then run and check.

```python
x = 10

def reads():
    print("reads:", x)

def shadows():
    x = 99
    print("shadows:", x)

reads()
shadows()
print("after:", x)
```

In [ ]:
# your prediction (as a comment):
# reads:    ?
# shadows:  ?
# after:    ?

# your code here


## Task 2 — Refactor away a global mutation

The code below uses a global `_running_total` that `add_value` mutates. This is exactly the pattern to avoid.

```python
_running_total = 0

def add_value(x):
    global _running_total
    _running_total += x
    return _running_total
```

Rewrite as a **pure** function `accumulate(values)` that takes a list of numbers and returns a list of running totals — no globals, no mutation of inputs.

Expected:
- `accumulate([1, 2, 3, 4])` → `[1, 3, 6, 10]`
- `accumulate([])` → `[]`

In [ ]:
def accumulate(values: list[float]) -> list[float]:
    pass  # TODO

assert accumulate([1, 2, 3, 4]) == [1, 3, 6, 10]
assert accumulate([]) == []
assert accumulate([10]) == [10]
print("ok")


## Task 3 — Mutating a global container is also a smell

Note: it's tempting to think "I'm not reassigning the global, I'm only mutating it — so I don't need `global`". Technically true, but it's still the same coupling problem.

Given this anti-pattern:

```python
log_lines = []

def log(msg):
    log_lines.append(msg)     # mutates module-level list
```

Rewrite `log` as `append_log(lines, msg)` that takes the list explicitly and returns the **new** list (don't mutate the input — return a fresh list).

```python
def append_log(lines: list[str], msg: str) -> list[str]:
    ...
```

Expected:
- `append_log([], "hello")` → `["hello"]`
- Calling it twice on `[]` returns the same thing — no shared state
- Original list is **not** mutated

In [ ]:
def append_log(lines: list[str], msg: str) -> list[str]:
    pass  # TODO

original = ["a"]
result = append_log(original, "b")
assert result == ["a", "b"]
assert original == ["a"]            # input unchanged
assert append_log([], "hello") == ["hello"]
assert append_log([], "hello") == ["hello"]   # independent, no shared state
print("ok")
